# Live-Action Drama Studio V2 — Zero-Fumble All-in-One ComfyUI (Krea-2 Turbo + MiniMax-H3)

Notebook sản xuất phim **Drama người thật** đã **kiểm tra binary `.safetensors` 100% tương thích AdaLN `[96768, 8]`** (không bao giờ lỗi shape mismatch trên base pruned!), chạy **100% official ComfyUI core (`0 custom nodes` mặc định)**, nối thẳng kho **`G:\My Drive\DramaStudio\`** trên Windows!

### Thư mục đồng bộ trên máy bạn (`G:\My Drive\DramaStudio\`)
- `inputs/` — Ảnh diễn viên, Character Sheet, Location Plate, WAV thoại Việt (`couple_female_ref.jpg`, `couple_male_ref.jpg`, `chuong_nhuoc_nam_clean.png` đã được đặt sẵn!).
- `outputs/keyframes/` — Ảnh keyframe **Krea-2 Turbo** (`+ photography LoRA / style_reference`) tự đồng bộ về máy.
- `outputs/videos/` — Video MP4 **MiniMax-H3** tự lưu thẳng lên Drive (không mất khi ngắt Colab!).
- `prompts/` — Workflow JSON đã lint chuẩn từ `h3_drama_cli.py` (**đã tạo sẵn 3 bài test A/B `ab01..ab03` để bấm chạy ngay sáng nay!**).
- `models/loras/` — (Tùy chọn) Bỏ thêm LoRA cá nhân `.safetensors` sẽ tự nạp vào ComfyUI.

### Ma trận Preset & Tương thích đã xác minh (A100-80GB)
| Preset | Độ phân giải | Steps / Sampler / Scheduler | Tốc độ ước tính | Dùng cho |
| :--- | :--- | :--- | :--- | :--- |
| **`draft_480p`** *(Mặc định)* | `864×480` → `RealESRGAN_x4plus` `1080p` | `20` / `res_multistep` / `simple` | ~2–5 phút | Tập phim dài nhiều cảnh thoại (`Ref2VA`) |
| **`turbo8_768p`** *(Nhanh 2.5×)* | Native `1280×736` (0 upscaler) | `8` / `euler` / `simple` (`shift 12/3`) | ~2–5 phút | MV, keyframe Krea-2 → `FL2VA/I2VA` (Kijai `v1.2` pruned), hoặc test nhanh `Ref2VA` (`v1.0`) |
| **`hero_736p`** *(Cảnh Đỉnh)* | Native `1280×736` (0 upscaler) | `20` / `res_multistep` / `simple` | ~6.5–13 phút | Cảnh cận mặt thân mật, thoại thì thầm 10s, tối đa lỗ chân lông & ánh mắt |

> **Nhắc nhở CU:** Chạy xong nhớ bấm **Runtime → Disconnect and delete runtime** ngay để giữ CU!


In [ ]:
# CELL 1: Mount Google Drive, verify GPU / PyTorch / CUDA & scratch SSD
import shutil, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")
for sub in ("inputs", "outputs/keyframes", "outputs/videos", "prompts", "prompts/batch", "models/loras"):
    (STUDIO_DIR / sub).mkdir(parents=True, exist_ok=True)

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_DIR = SCRATCH_ROOT / "ComfyUI"

print("=== GPU & PYTORCH ===")
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True
).strip())
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: {torch.cuda.is_bf16_supported()}")

usage = shutil.disk_usage(SCRATCH_ROOT)
print(f"=== SCRATCH DISK ({SCRATCH_ROOT}) ===")
print(f"Free: {usage.free / 1e9:.1f} GB / Total: {usage.total / 1e9:.1f} GB")
print(f"=== GOOGLE DRIVE STUDIO ===\n{STUDIO_DIR} (exists={STUDIO_DIR.exists()})")
print("Inputs :", sorted(p.name for p in (STUDIO_DIR / "inputs").iterdir())[:20])
print("Prompts:", sorted(p.name for p in (STUDIO_DIR / "prompts").glob("*.json"))[:20])


In [ ]:
# CELL 2: Install Official ComfyUI Core (0 custom nodes by default) + Fast HF Downloader
INSTALL_KREA2_IDENTITY_EDIT_NODE = False  # Set True ONLY if you want conradlocke/krea2-identity-edit V2 restaging node

import subprocess, sys
from pathlib import Path

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_DIR = SCRATCH_ROOT / "ComfyUI"

if not COMFY_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(COMFY_DIR), "pull", "--ff-only"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_transfer]", "av", "imageio-ffmpeg"], check=True)

if INSTALL_KREA2_IDENTITY_EDIT_NODE:
    node_dir = COMFY_DIR / "custom_nodes" / "comfyui-krea2edit"
    if not node_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lbouaraba/comfyui-krea2edit.git", str(node_dir)], check=True)
        req = node_dir / "requirements.txt"
        if req.exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    print("Installed optional node: comfyui-krea2edit")

ver_file = COMFY_DIR / "comfyui_version.py"
print("ComfyUI ready at:", COMFY_DIR)
if ver_file.exists():
    print(ver_file.read_text()[:200].strip())


In [ ]:
# CELL 3: Parallel High-Speed Model Download (Binary-Verified 100% Pruned-Compatible!)
INSTALL_KREA2 = True                 # ~19.5 GB : Krea-2 Turbo Int8 + Qwen3-VL 4B + VAE + Style Reference LoRA
INSTALL_KREA2_PHOTO_LORAS = True     # ~918 MB  : Anti-plastic East Asian & 35mm Analog Film LoRAs for Krea-2 Turbo
INSTALL_REF2VA = True                # ~40.1 GB : MiniMax-H3 Ref2VA Pruned Int8 + Qwen3-VL 32B NVFP4 + VAEs + 10 Embeddings
INSTALL_FL2VA = True                 # ~21.0 GB : MiniMax-H3 FL2VA Pruned Int8 (best for Krea-2 keyframe -> I2VA/FL2VA!)
INSTALL_TURBO_LORAS = True           # ~2.3 GB  : Kijai pruned FL2V 8-step v1.2 (335MB) + LightX2V Ref2V 8-step v1.0 (1.96GB)
INSTALL_REALISM_LORA = True          # ~131 MB  : fal/MiniMax-H3-Realism-People-LoRA (trigger: r34l1sm, 0 AdaLN = 100% safe!)
INSTALL_SINGULARITY_V2 = False       # ~21.0 GB : Optional WarmBloodAban/Minimax-h3_Singularity V2 Pruned Int8 (East Asian realism)
INSTALL_LATENT_UPSCALER = False      # ~364 MB  : Optional LBH-123-AI native 2D latent upscaler v1
INSTALL_KREA2_IDENTITY_EDIT = False  # ~914 MB  : Optional conradlocke/krea2-identity-edit V2 r64 LoRAs (needs Cell 2 toggle)

import os, shutil, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
MODELS = SCRATCH_ROOT / "ComfyUI" / "models"
STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")

jobs = []  # (repo_id, repo_filename, target_rel_dir, rename_or_None)

if INSTALL_KREA2:
    jobs += [
        ("Comfy-Org/Krea-2", "diffusion_models/krea2_turbo_int8_convrot.safetensors", "diffusion_models", None),
        ("Comfy-Org/Krea-2", "text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "text_encoders", None),
        ("Comfy-Org/Krea-2", "vae/qwen_image_vae.safetensors", "vae", None),
        ("Comfy-Org/Krea-2", "loras/krea2_style_reference.safetensors", "loras", None),
    ]

if INSTALL_KREA2_PHOTO_LORAS:
    jobs += [
        ("Omnico/Krea2_turbo_diff_loras", "TherealrealKrea2Max_maxArtAsian_bf16_r64.safetensors", "loras", None),
        ("Omnico/Krea2_turbo_diff_loras", "analogMadnessKrea2Turbo_v4_bf16_r64.safetensors", "loras", None),
    ]

if INSTALL_KREA2_IDENTITY_EDIT:
    jobs += [
        ("conradlocke/krea2-identity-edit", "krea2_identity_edit_v2_r64.safetensors", "loras", None),
        ("conradlocke/krea2-identity-edit", "krea2_general_edit_v2_r64.safetensors", "loras", None),
    ]

if INSTALL_REF2VA or INSTALL_FL2VA or INSTALL_SINGULARITY_V2:
    jobs += [
        ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders", None),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae", None),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae", None),
    ]
    for emb in [
        "minimaxh3_art_is_explosion.safetensors",
        "minimaxh3_blooming_flowers.safetensors",
        "minimaxh3_bullet_time.safetensors",
        "minimaxh3_dark_magic.safetensors",
        "minimaxh3_fire_breath.safetensors",
        "minimaxh3_four_seasons.safetensors",
        "minimaxh3_kiss_camera.safetensors",
        "minimaxh3_spiral_ascent.safetensors",
        "minimaxh3_storm_magic.safetensors",
        "minimaxh3_truman_show.safetensors",
    ]:
        jobs.append(("Comfy-Org/MiniMax-H3", f"embeddings/{emb}", "embeddings", None))

if INSTALL_REF2VA:
    jobs.append(("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "diffusion_models", None))

if INSTALL_FL2VA:
    jobs.append(("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models", None))

if INSTALL_SINGULARITY_V2:
    jobs.append(("WarmBloodAban/Minimax-h3_Singularity", "Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors", "diffusion_models", None))

if INSTALL_TURBO_LORAS:
    jobs += [
        # Binary-verified [96768, 8] AdaLN exact match for pruned bases:
        ("Kijai/MiniMax-H3_comfy", "loras/minimax_h3_fl2v_turbo_8step_v1.2_768p_pruned_avg_rank_24_bf16.safetensors", "loras", None),
        # Binary-verified 0 AdaLN tensors = 100% pruned-safe (community-preferred v1.0):
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors", "loras", None),
    ]

if INSTALL_REALISM_LORA:
    jobs.append(("fal/MiniMax-H3-Realism-People-LoRA", "h3-realism-people-t2v-i2v-r2v.safetensors", "loras", None))

if INSTALL_LATENT_UPSCALER:
    jobs.append((
        "LBH-123-AI/Minimax_h3_latent_Upscaler",
        "minimax_h3_latent_upscaler_2d_conv_v1/minimax_h3_latent_upscaler_2d_conv_v1_bf16.safetensors",
        "latent_upscale_models",
        None,
    ))

def download_one(item):
    repo_id, repo_file, rel_dir, rename = item
    target_dir = MODELS / rel_dir
    target_dir.mkdir(parents=True, exist_ok=True)
    dst = target_dir / (rename or Path(repo_file).name)
    if dst.exists() and dst.stat().st_size > 1024:
        return f"SKIP (cached): {dst.name}"
    cached = hf_hub_download(repo_id=repo_id, filename=repo_file)
    shutil.copy2(cached, dst)
    return f"OK ({dst.stat().st_size/1e6:.1f} MB): {dst.name}"

with ThreadPoolExecutor(max_workers=4) as ex:
    for msg in ex.map(download_one, jobs):
        print(msg, flush=True)

# Always install live-action RealESRGAN_x4plus.pth (NEVER anime_6B for live-action!)
up_dir = MODELS / "upscale_models"
up_dir.mkdir(parents=True, exist_ok=True)
realesrgan = up_dir / "RealESRGAN_x4plus.pth"
if not realesrgan.exists():
    urllib.request.urlretrieve(
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        str(realesrgan),
    )
print(f"OK ({realesrgan.stat().st_size/1e6:.1f} MB): {realesrgan.name}")

# Sync custom LoRAs from G:\\My Drive\\DramaStudio\\models\\loras\\
user_loras = STUDIO_DIR / "models" / "loras"
if user_loras.exists():
    lora_dst = MODELS / "loras"
    lora_dst.mkdir(parents=True, exist_ok=True)
    for f in user_loras.glob("*.safetensors"):
        dst = lora_dst / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
            print(f"SYNCED USER LORA: {f.name}")

print("ALL MODELS READY!")


In [ ]:
# CELL 4: Launch ComfyUI Server (127.0.0.1:8188) + Optional Custom-Domain Cloudflare Named Tunnel
import os, subprocess, sys, time, urllib.request
from pathlib import Path

# OPTIONAL: Paste your Cloudflare Named Tunnel token if you want Web UI on your own domain.
CLOUDFLARE_TUNNEL_TOKEN = ""

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_DIR = SCRATCH_ROOT / "ComfyUI"
LOG_FILE = Path("/tmp/comfy_server.log")

def server_alive():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if not server_alive():
    log_fh = LOG_FILE.open("w", encoding="utf-8")
    subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
        cwd=str(COMFY_DIR),
        stdout=log_fh,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(90):
        if server_alive():
            break
        time.sleep(1)

assert server_alive(), f"ComfyUI failed to start! Tail:\n{LOG_FILE.read_text(errors='replace')[-2000:]}"
print("COMFYUI SERVER LIVE AT http://127.0.0.1:8188")

if CLOUDFLARE_TUNNEL_TOKEN.strip():
    cf_bin = Path("/usr/local/bin/cloudflared")
    if not cf_bin.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(cf_bin),
        )
        os.chmod(cf_bin, 0o755)
    cf_log = Path("/tmp/cloudflared.log").open("w")
    subprocess.Popen(
        [str(cf_bin), "tunnel", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN.strip()],
        stdout=cf_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print("CUSTOM-DOMAIN CLOUDFLARE NAMED TUNNEL STARTED!")


In [ ]:
# CELL 5: Stage 1 — Krea-2 Turbo Keyframe & Style-Locked Plate Helpers (~3s/image on A100!)
# Saves directly to G:\\My Drive\\DramaStudio\\outputs\\keyframes\\
import json, shutil, time, urllib.parse, urllib.request, uuid
from pathlib import Path

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_INPUT = SCRATCH_ROOT / "ComfyUI" / "input"
STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")
KEYFRAME_OUT = STUDIO_DIR / "outputs" / "keyframes"
SERVER = "http://127.0.0.1:8188"

def _run_image_wf(wf, out_name):
    req = urllib.request.Request(
        f"{SERVER}/prompt",
        data=json.dumps({"prompt": wf, "client_id": str(uuid.uuid4())}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    pid = json.loads(urllib.request.urlopen(req, timeout=30).read().decode("utf-8"))["prompt_id"]
    for _ in range(180):
        hist = json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}", timeout=20).read().decode("utf-8"))
        if pid in hist:
            item = hist[pid]
            if item.get("status", {}).get("status_str") == "error":
                raise RuntimeError(f"Krea-2 error: {item['status']}")
            for no in item.get("outputs", {}).values():
                for img in no.get("images", []):
                    qs = urllib.parse.urlencode({"filename": img["filename"], "subfolder": img.get("subfolder", ""), "type": img.get("type", "output")})
                    dst = KEYFRAME_OUT / (out_name if out_name.endswith(".png") else f"{out_name}.png")
                    dst.write_bytes(urllib.request.urlopen(f"{SERVER}/view?{qs}", timeout=60).read())
                    print(f"SAVED KEYFRAME TO DRIVE: {dst} ({dst.stat().st_size/1e6:.2f} MB)")
                    return dst
        time.sleep(1)
    raise RuntimeError("Krea-2 timed out")

def generate_krea2_keyframe(
    prompt: str,
    out_name: str,
    width: int = 1280,
    height: int = 736,
    seed: int = 42001,
    steps: int = 8,
    photo_lora: str = "analogMadnessKrea2Turbo_v4_bf16_r64.safetensors",  # or "TherealrealKrea2Max_maxArtAsian_bf16_r64.safetensors" or ""
    photo_lora_strength: float = 0.75,
):
    wf = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "krea2_turbo_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": prompt}},
        "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
        "6": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
    }
    model_ref = ["1", 0]
    if photo_lora:
        wf["7"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": model_ref, "lora_name": photo_lora, "strength_model": photo_lora_strength}}
        model_ref = ["7", 0]
    wf["8"] = {
        "class_type": "KSampler",
        "inputs": {
            "model": model_ref, "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0],
            "seed": seed, "steps": steps, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
        },
    }
    wf["9"] = {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}}
    wf["10"] = {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": Path(out_name).stem}}
    return _run_image_wf(wf, out_name)

def generate_krea2_style_ref(
    prompt: str,
    style_image_name: str,   # file inside DramaStudio/inputs/ or DramaStudio/outputs/keyframes/
    out_name: str,
    width: int = 1280,
    height: int = 736,
    seed: int = 42002,
    steps: int = 8,
):
    # Exact official Comfy-Org Krea-2 Image Style Reference workflow (locks color grade/lighting across shots!)
    for d in (STUDIO_DIR / "inputs", KEYFRAME_OUT):
        cand = d / style_image_name
        if cand.exists():
            shutil.copy2(cand, COMFY_INPUT / style_image_name)
            break
    wf = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "krea2_turbo_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
        "4": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": "krea2_style_reference.safetensors", "strength_model": 1.0}},
        "5": {"class_type": "ModelSamplingFlux", "inputs": {"model": ["4", 0], "max_shift": 1.15, "base_shift": 0.5, "width": width, "height": height}},
        "6": {"class_type": "LoadImage", "inputs": {"image": style_image_name}},
        "7": {"class_type": "TextEncodeQwenImageEditPlus", "inputs": {"clip": ["2", 0], "vae": ["3", 0], "image1": ["6", 0], "prompt": prompt}},
        "8": {"class_type": "FluxKontextMultiReferenceLatentMethod", "inputs": {"conditioning": ["7", 0], "reference_latents_method": "index_timestep_zero"}},
        "9": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["8", 0]}},
        "10": {"class_type": "CFGGuider", "inputs": {"model": ["5", 0], "positive": ["8", 0], "negative": ["9", 0], "cfg": 1.0}},
        "11": {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}},
        "12": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
        "13": {"class_type": "BasicScheduler", "inputs": {"model": ["5", 0], "scheduler": "simple", "steps": steps, "denoise": 1.0}},
        "14": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
        "15": {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["11", 0], "guider": ["10", 0], "sampler": ["12", 0], "sigmas": ["13", 0], "latent_image": ["14", 0]}},
        "16": {"class_type": "VAEDecode", "inputs": {"samples": ["15", 0], "vae": ["3", 0]}},
        "17": {"class_type": "SaveImage", "inputs": {"images": ["16", 0], "filename_prefix": Path(out_name).stem}},
    }
    return _run_image_wf(wf, out_name)

print("Krea-2 Turbo & Style-Reference helpers ready!")


In [ ]:
# CELL 6: Stage 2 — Queue Any Validated MiniMax-H3 Workflow JSON from DramaStudio/prompts/ (Non-Blocking!)
import json, shutil, time, urllib.request, uuid
from pathlib import Path

# Pre-staged morning A/B tests ready right now:
#   "ab01_couple_turbo8_ref2va.json"          (8-step Turbo 736p vs 20-step couple_intimate_10s)
#   "ab02_couple_turbo8_realism_ref2va.json"  (8-step Turbo + fal Realism People LoRA r34l1sm)
#   "ab03_rooftop_draft480_realesrgan.json"   (draft_480p -> RealESRGAN_x4plus 1080p vs native 736p)
WORKFLOW_JSON_NAME = "ab01_couple_turbo8_ref2va.json"

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_INPUT = SCRATCH_ROOT / "ComfyUI" / "input"
STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")
SERVER = "http://127.0.0.1:8188"

wf_path = STUDIO_DIR / "prompts" / WORKFLOW_JSON_NAME
assert wf_path.exists(), f"Missing {wf_path}. Available: {sorted(p.name for p in (STUDIO_DIR / 'prompts').glob('*.json'))}"

# Sync inputs + generated keyframes into ComfyUI/input/
for src_dir in (STUDIO_DIR / "inputs", STUDIO_DIR / "outputs" / "keyframes"):
    for f in src_dir.iterdir():
        if f.is_file():
            dst = COMFY_INPUT / f.name
            if not dst.exists() or dst.stat().st_size != f.stat().st_size:
                shutil.copy2(f, dst)

raw = json.loads(wf_path.read_text(encoding="utf-8-sig"))
wf = raw["workflow"] if "workflow" in raw else raw
meta = raw.get("meta", {})

req = urllib.request.Request(
    f"{SERVER}/prompt",
    data=json.dumps({"prompt": wf, "client_id": str(uuid.uuid4())}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
pid = json.loads(urllib.request.urlopen(req, timeout=30).read().decode("utf-8"))["prompt_id"]
Path("/tmp/active_h3_job.json").write_text(json.dumps({"prompt_id": pid, "name": wf_path.stem, "meta": meta}, indent=2))
print(f"QUEUED {wf_path.name} ({meta.get('preset')} {meta.get('width')}x{meta.get('height')} {meta.get('steps')}steps) -> pid={pid}", flush=True)
time.sleep(25)
print("--- COMFY LOG TAIL ---\n" + "\n".join(Path("/tmp/comfy_server.log").read_text(errors="replace").splitlines()[-8:]))


In [ ]:
# CELL 7: Stage 3 — Check Status & Save Completed MP4 Directly to G:\My Drive\DramaStudio\outputs\videos\
import json, subprocess, urllib.parse, urllib.request
from pathlib import Path

STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")
OUT_VIDEOS = STUDIO_DIR / "outputs" / "videos"
SERVER = "http://127.0.0.1:8188"

job_file = Path("/tmp/active_h3_job.json")
assert job_file.exists(), "No active job recorded — run Cell 6 first."
job = json.loads(job_file.read_text())
pid = job["prompt_id"]

hist = json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}", timeout=20).read().decode("utf-8"))
if pid not in hist:
    print("STILL RUNNING... Latest ComfyUI progress:")
    print("\n".join(Path("/tmp/comfy_server.log").read_text(errors="replace").splitlines()[-6:]))
else:
    item = hist[pid]
    if item.get("status", {}).get("status_str") == "error":
        raise RuntimeError(f"ComfyUI FAILED: {item['status']}")
    saved = None
    for no in item.get("outputs", {}).values():
        for m in (no.get("videos") or no.get("gifs") or no.get("images") or []):
            if str(m.get("filename", "")).lower().endswith((".mp4", ".webm", ".mkv")):
                qs = urllib.parse.urlencode({"filename": m["filename"], "subfolder": m.get("subfolder", ""), "type": m.get("type", "output")})
                dst = OUT_VIDEOS / f"{job['name']}.mp4"
                dst.write_bytes(urllib.request.urlopen(f"{SERVER}/view?{qs}", timeout=180).read())
                saved = dst
                break
        if saved:
            break
    assert saved, f"Completed but no MP4 found in outputs for {pid}"
    print(f"SAVED TO GOOGLE DRIVE: {saved} ({saved.stat().st_size/1e6:.2f} MB)")
    print(subprocess.check_output(
        ["ffprobe", "-v", "error", "-show_entries", "stream=codec_type,codec_name,width,height,r_frame_rate,sample_rate,duration", "-of", "compact", str(saved)],
        text=True
    ).strip())


In [ ]:
# CELL 8: Optional Detached Batch Runner (Renders all JSONs in DramaStudio/prompts/batch/ sequentially -> Drive)
import subprocess, sys
from pathlib import Path

BATCH_WORKER = Path("/tmp/h3_batch_worker.py")
BATCH_WORKER.write_text('''import json, shutil, time, urllib.parse, urllib.request, uuid
from pathlib import Path

SCRATCH_ROOT = Path("/mnt/local-scratch") if Path("/mnt/local-scratch").exists() else Path("/content")
COMFY_INPUT = SCRATCH_ROOT / "ComfyUI" / "input"
STUDIO_DIR = Path("/content/drive/MyDrive/DramaStudio")
BATCH_DIR = STUDIO_DIR / "prompts" / "batch"
OUT_VIDEOS = STUDIO_DIR / "outputs" / "videos"
SERVER = "http://127.0.0.1:8188"

for src_dir in (STUDIO_DIR / "inputs", STUDIO_DIR / "outputs" / "keyframes"):
    if src_dir.exists():
        for f in src_dir.iterdir():
            if f.is_file():
                dst = COMFY_INPUT / f.name
                if not dst.exists() or dst.stat().st_size != f.stat().st_size:
                    shutil.copy2(f, dst)

wf_files = sorted(BATCH_DIR.glob("*.json")) if BATCH_DIR.exists() else []
print(f"Batch queue: {len(wf_files)} workflow(s)", flush=True)

for idx, wfp in enumerate(wf_files, 1):
    out_mp4 = OUT_VIDEOS / f"{wfp.stem}.mp4"
    if out_mp4.exists() and out_mp4.stat().st_size > 100_000:
        print(f"[{idx}/{len(wf_files)}] SKIP existing: {out_mp4.name}", flush=True)
        continue
    raw = json.loads(wfp.read_text(encoding="utf-8-sig"))
    wf = raw["workflow"] if "workflow" in raw else raw
    t0 = time.time()
    req = urllib.request.Request(
        f"{SERVER}/prompt",
        data=json.dumps({"prompt": wf, "client_id": str(uuid.uuid4())}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    pid = json.loads(urllib.request.urlopen(req, timeout=30).read().decode("utf-8"))["prompt_id"]
    print(f"[{idx}/{len(wf_files)}] START {wfp.name} (pid={pid})", flush=True)
    while True:
        hist = json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}", timeout=20).read().decode("utf-8"))
        if pid in hist:
            item = hist[pid]
            if item.get("status", {}).get("status_str") == "error":
                print(f"[{idx}/{len(wf_files)}] ERROR on {wfp.name}: {item['status']}", flush=True)
                break
            for no in item.get("outputs", {}).values():
                for m in (no.get("videos") or no.get("gifs") or no.get("images") or []):
                    if str(m.get("filename", "")).lower().endswith((".mp4", ".webm", ".mkv")):
                        qs = urllib.parse.urlencode({"filename": m["filename"], "subfolder": m.get("subfolder", ""), "type": m.get("type", "output")})
                        out_mp4.write_bytes(urllib.request.urlopen(f"{SERVER}/view?{qs}", timeout=180).read())
                        print(f"[{idx}/{len(wf_files)}] DONE {out_mp4.name} ({out_mp4.stat().st_size/1e6:.2f} MB in {time.time()-t0:.0f}s)", flush=True)
                        break
            break
        time.sleep(5)
print("BATCH_ALL_DONE", flush=True)
''', encoding="utf-8")

# Uncomment to launch detached batch worker:
# log = Path("/tmp/h3_batch.log").open("w")
# subprocess.Popen([sys.executable, str(BATCH_WORKER)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("Batch worker prepared at /tmp/h3_batch_worker.py")
